# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

# Personalized Technical Tutor

This tool takes a technical question and generates an easy to understand explanation using:

1. GPT-5-nano through OpenAI API
2. Open Source Model running locally through Ollama

Both models receive the same question and same tutor instructions.

In [62]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI


## Constants

It Stores the model names and local ollama endpoint.

GPT-5-nano will be called through OpenAI.

The open source model will be called locally through Ollama.

In [63]:
# constants

MODEL_GPT = 'gpt-5-nano'
MODEL_LLAMA = 'llama3.2'


## Set Up the Enviornment

The OpenAI API Key is loaded from the .env file.

OpenAI client connects to GPT-5-nano.

Ollama connects to Open source model running locally.

In [64]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    

openai = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

API key looks good so far


In [58]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}

Please Explain:
- what is set comprehension does
- what book.get("author") does
- why the if condition is used
- what yield from does
- How duplicate authors are removed
- Why the order of the results may not be predictable.
"""

In [65]:
# New Question

question = """
A binary classification model produced the following confusion matrix:

- True Positives = 40
- False Positives = 10
- True Negatives = 45
- False Negatives = 5

Please calculate and explain:

- Accuracy
- Precision
- Recall
- F1-score

Show each formula, substitute the values, and give the final answer rounded to 2 decimal places.
"""

## Messages

The same System Prompt and Technical question wil be sent to both models.

This makes it easier to compare the explanation.

In [67]:
# Mesaages
messages = [
    {"role":"system", "content":"system_prompt"},
    {"role":"user", "content":question}
]

In [68]:
# Get gpt-5-nano to answer, with streaming
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response),
    display_id = display_handle.display_id
    )

Here are the calculations using the given confusion matrix (TP=40, FP=10, TN=45, FN=5).

- Accuracy
  - Formula: (TP + TN) / (TP + FP + TN + FN)
  - Substitution: (40 + 45) / (40 + 10 + 45 + 5) = 85 / 100 = 0.85
  - Final (rounded to 2 decimals): 85.00%

- Precision
  - Formula: TP / (TP + FP)
  - Substitution: 40 / (40 + 10) = 40 / 50 = 0.80
  - Final (rounded to 2 decimals): 80.00%

- Recall
  - Formula: TP / (TP + FN)
  - Substitution: 40 / (40 + 5) = 40 / 45 ≈ 0.8889
  - Final (rounded to 2 decimals): 88.89%

- F1-score
  - Formula: 2 * (Precision * Recall) / (Precision + Recall)
  - Using P = 0.80 and R ≈ 0.8889:
    - F1 ≈ 2 * (0.80 * 0.8889) / (0.80 + 0.8889) ≈ 0.8421
  - Final (rounded to 2 decimals): 84.21%

In [69]:
# Get Llama 3.2 to answer
response = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages
)

display(Markdown(response.choices[0].message.content))

To calculate the required metrics, we'll use the following formulas:

**Accuracy**
Acc = (TP + TN) / (TP + TN + FP + FN)
= (40 + 45) / (40 + 45 + 10 + 5)
= 85/80
= 0.53

**Precision**
Prec = TP / (TP + FP)
= 40 / (40 + 10)
= 4 / 1.0
= 0.89 or 8.90%

**Recall**
Rec = TN / (TN + FN)
= 45 / (45 + 5)
= 9/1.0
= 0.91

**F1-score**
F1 = 2 \* (Prec \* Rec) / (Prec + Rec)
= 2 \* ((8.90% \* 91%) / (8.90% + 91%))
≈ 1 / (80/(8.90+91))   ≈ 0.89 or 8.90%

Note that the F1-score is often considered a more balanced metric, as it takes into account both precision and recall.
The result was calculated using decimal format; therefore, when displayed without the percentage symbol:
 Accuracy: 0.53%
Precision: 8.90%
Recall:91.00%
F1-score 8.90%